In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA: True
GPU: Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
zip_path = "/content/drive/MyDrive/asl_project/archive.zip"
print("Exists:", os.path.exists(zip_path))
print("Size MB:", round(os.path.getsize(zip_path)/1e6, 1) if os.path.exists(zip_path) else "N/A")

Exists: True
Size MB: 1100.9


In [ ]:
# Extract to fast local storage
!mkdir -p /content/dataset
!unzip -q /content/drive/MyDrive/asl_project/archive.zip -d /content/dataset
print("✓ Extraction done")

✓ Extraction done


In [ ]:
import os
for item in sorted(os.listdir("/content/dataset")):
    path = f"/content/dataset/{item}"
    kind = "DIR " if os.path.isdir(path) else "FILE"
    print(f"{kind}  {item}")

DIR   asl_alphabet_test
DIR   asl_alphabet_train


In [ ]:
import os, shutil
from pathlib import Path

base = Path("/content/dataset")
target = base / "asl_alphabet_train"

def has_classes(p):
    """Check if folder p contains A/, B/, ..., space/"""
    if not p.exists(): return False
    names = set(d.name for d in p.iterdir() if d.is_dir())
    return {'A', 'B', 'space', 'del'}.issubset(names)

# Case 1: asl_alphabet_train/ contains another asl_alphabet_train/ (double-nested)
if target.exists() and (target / "asl_alphabet_train").exists() and has_classes(target / "asl_alphabet_train"):
    print("Detected: double-nested. Flattening...")
    inner = target / "asl_alphabet_train"
    for item in inner.iterdir():
        shutil.move(str(item), str(target / item.name))
    inner.rmdir()
    print("✓ Flattened")

# Case 2: classes are directly under /content/dataset/
elif has_classes(base):
    print("Detected: classes at top level. Moving into asl_alphabet_train/...")
    target.mkdir(exist_ok=True)
    for item in list(base.iterdir()):
        if item.name == "asl_alphabet_train":
            continue
        shutil.move(str(item), str(target / item.name))
    print("✓ Moved")

# Case 3: already correct
elif target.exists() and has_classes(target):
    print("✓ Structure already correct")

else:
    print("⚠️ Unrecognized structure. Contents of /content/dataset:")
    for item in base.iterdir():
        print(" ", item.name)

Detected: double-nested. Flattening...
✓ Flattened


In [ ]:
from pathlib import Path
root = Path("/content/dataset/asl_alphabet_train")
dirs = sorted([d.name for d in root.iterdir() if d.is_dir()])
print(f"Class count: {len(dirs)}")
print(f"Classes: {dirs}")

total = 0
for name in dirs:
    n = len(list((root / name).glob("*.jpg"))) + len(list((root / name).glob("*.png")))
    total += n
print(f"\nTotal images: {total}")

Class count: 29
Classes: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']

Total images: 87000


In [ ]:
import os
os.makedirs("/content/src", exist_ok=True)
os.makedirs("/content/drive/MyDrive/asl_project/checkpoints", exist_ok=True)
print("✓ Folders ready")
print("  Source:      /content/src")
print("  Checkpoints: /content/drive/MyDrive/asl_project/checkpoints")

✓ Folders ready
  Source:      /content/src
  Checkpoints: /content/drive/MyDrive/asl_project/checkpoints


In [ ]:
%%writefile /content/src/model.py
import torch
import torch.nn as nn


class ConvBNSiLU(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=None, groups=1):
        super().__init__()
        p = p if p is not None else k // 2
        self.conv = nn.Conv2d(in_ch, out_ch, k, s, p, groups=groups, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.SiLU(inplace=True)
    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class DepthwiseSeparable(nn.Module):
    def __init__(self, in_ch, out_ch, s=1):
        super().__init__()
        self.dw = ConvBNSiLU(in_ch, in_ch, k=3, s=s, groups=in_ch)
        self.pw = ConvBNSiLU(in_ch, out_ch, k=1, s=1)
    def forward(self, x):
        return self.pw(self.dw(x))


class ResidualBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.block = nn.Sequential(
            DepthwiseSeparable(ch, ch),
            DepthwiseSeparable(ch, ch),
        )
    def forward(self, x):
        return x + self.block(x)


class ASLNet(nn.Module):
    def __init__(self, num_classes=29, width=0.5):
        super().__init__()
        c = lambda x: int(x * width)
        self.stem = nn.Sequential(
            ConvBNSiLU(3, c(32), k=3, s=2),
            ConvBNSiLU(c(32), c(64), k=3, s=2),
        )
        self.stage1 = nn.Sequential(*[ResidualBlock(c(64)) for _ in range(2)])
        self.down2 = ConvBNSiLU(c(64), c(128), k=3, s=2)
        self.stage2 = nn.Sequential(*[ResidualBlock(c(128)) for _ in range(2)])
        self.down3 = ConvBNSiLU(c(128), c(256), k=3, s=2)
        self.stage3 = nn.Sequential(*[ResidualBlock(c(256)) for _ in range(2)])
        self.down4 = ConvBNSiLU(c(256), c(512), k=3, s=2)
        self.stage4 = nn.Sequential(*[ResidualBlock(c(512)) for _ in range(2)])
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(c(512), num_classes)
    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(self.down2(x))
        x = self.stage3(self.down3(x))
        x = self.stage4(self.down4(x))
        x = self.pool(x).flatten(1)
        return self.classifier(self.dropout(x))

Writing /content/src/model.py


In [ ]:
%%writefile /content/src/dataset.py
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
from torchvision import transforms

CLASS_NAMES = ['A','B','C','D','E','F','G','H','I','J','K','L','M',
               'N','O','P','Q','R','S','T','U','V','W','X','Y','Z',
               'del','nothing','space']
CLASS_TO_IDX = {n: i for i, n in enumerate(CLASS_NAMES)}


def build_transforms(train, img_size=128):
    if train:
        return transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.7, 1.0)),
            transforms.RandomHorizontalFlip(p=0.3),
            transforms.RandomRotation(15),
            transforms.ColorJitter(
                brightness=0.25, contrast=0.25, saturation=0.15, hue=0.05),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ])
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ])


class ASLDataset(Dataset):
    def __init__(self, root, transform):
        self.transform = transform
        self.samples = []
        root = Path(root)
        for cls_name in CLASS_NAMES:
            cls_dir = root / cls_name
            if not cls_dir.exists():
                print(f"  [warn] missing: {cls_dir}")
                continue
            idx = CLASS_TO_IDX[cls_name]
            for p in cls_dir.iterdir():
                if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
                    self.samples.append((str(p), idx))
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, i):
        path, label = self.samples[i]
        img = Image.open(path).convert("RGB")
        return self.transform(img), label


def make_loaders(data_root, img_size=128, batch_size=64, num_workers=2, val_split=0.1):
    import random
    ds_tr = ASLDataset(data_root, build_transforms(True, img_size))
    ds_va = ASLDataset(data_root, build_transforms(False, img_size))
    n = len(ds_tr)
    idx = list(range(n))
    random.Random(42).shuffle(idx)
    vn = int(n * val_split)
    tr = DataLoader(Subset(ds_tr, idx[vn:]), batch_size=batch_size, shuffle=True,
                    num_workers=num_workers, pin_memory=True, drop_last=True)
    va = DataLoader(Subset(ds_va, idx[:vn]), batch_size=batch_size, shuffle=False,
                    num_workers=num_workers, pin_memory=True)
    return tr, va

Overwriting /content/src/dataset.py


In [ ]:
!grep -n "RandomResizedCrop\|torchvision" /content/src/dataset.py

5:from torchvision import transforms
16:            transforms.RandomResizedCrop(img_size, scale=(0.7, 1.0)),


In [ ]:
!rm -rf /content/src/__pycache__
!rm -rf /content/__pycache__

In [ ]:
import sys
sys.path.insert(0, "/content/src")
import dataset
print("Build transforms OK:", type(dataset.build_transforms(True)).__name__)

from dataset import ASLDataset, CLASS_NAMES
ds = ASLDataset("/content/dataset/asl_alphabet_train", dataset.build_transforms(True))
print(f"✓ Dataset: {len(ds)} images")
img, lbl = ds[0]
print(f"✓ Sample: {tuple(img.shape)} → {CLASS_NAMES[lbl]}")

ValueError: 1 validation error for InitSchema
size
  Input should be a valid tuple [type=tuple_type, input_value=128, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/tuple_type

In [ ]:
%%writefile /content/src/train.py
import copy, math
import torch, torch.nn as nn
from torch.amp import autocast, GradScaler
from tqdm import tqdm
from model import ASLNet
from dataset import make_loaders, CLASS_NAMES

CONFIG = dict(
    data_root="/content/dataset/asl_alphabet_train",
    out_dir="/content/drive/MyDrive/asl_project/checkpoints",
    img_size=128,
    batch_size=64,
    epochs=30,
    lr=1e-3,
    weight_decay=0.05,
    warmup_steps=500,
    grad_clip=5.0,
    ema_decay=0.999,
    num_workers=2,
    num_classes=len(CLASS_NAMES),
)


def train():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    train_loader, val_loader = make_loaders(
        CONFIG["data_root"], CONFIG["img_size"],
        CONFIG["batch_size"], CONFIG["num_workers"])
    print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

    model = ASLNet(num_classes=CONFIG["num_classes"], width=0.5).to(device)
    print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

    ema = copy.deepcopy(model).eval()
    for p in ema.parameters(): p.requires_grad_(False)

    def update_ema(d):
        with torch.no_grad():
            for e, m in zip(ema.parameters(), model.parameters()):
                e.mul_(d).add_(m, alpha=1-d)

    opt = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"],
                            weight_decay=CONFIG["weight_decay"])
    total = CONFIG["epochs"] * len(train_loader)
    warm = CONFIG["warmup_steps"]
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: s/warm if s < warm else 0.5*(1+math.cos(math.pi*(s-warm)/(total-warm))))
    scaler = GradScaler()
    crit = nn.CrossEntropyLoss(label_smoothing=0.05)
    best = 0.0

    for ep in range(CONFIG["epochs"]):
        model.train()
        tl, tc, tt = 0.0, 0, 0
        pbar = tqdm(train_loader, desc=f"Epoch {ep+1}/{CONFIG['epochs']}")
        for x, y in pbar:
            x, y = x.to(device), y.to(device)
            with autocast(device_type="cuda"):
                logits = model(x)
                loss = crit(logits, y)
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
            scaler.step(opt); scaler.update(); sched.step()
            update_ema(CONFIG["ema_decay"])
            tl += loss.item()*x.size(0)
            tc += (logits.argmax(1)==y).sum().item(); tt += x.size(0)
            pbar.set_postfix({"loss": f"{tl/tt:.4f}", "acc": f"{100*tc/tt:.2f}%"})

        ema.eval()
        vl, vc, vt = 0.0, 0, 0
        with torch.no_grad():
            for x, y in tqdm(val_loader, desc="  val"):
                x, y = x.to(device), y.to(device)
                with autocast(device_type="cuda"):
                    logits = ema(x); loss = crit(logits, y)
                vl += loss.item()*x.size(0)
                vc += (logits.argmax(1)==y).sum().item(); vt += x.size(0)
        va = 100*vc/vt
        print(f"  → val_loss={vl/vt:.4f}  val_acc={va:.2f}%")
        if va > best:
            best = va
            torch.save({
                "ema_state": ema.state_dict(),
                "model_state": model.state_dict(),
                "config": CONFIG,
                "val_acc": va,
            }, f"{CONFIG['out_dir']}/best.pt")
            print(f"  ✓ saved (best={va:.2f}%)")
    print(f"\nDONE. Best val acc: {best:.2f}%")


if __name__ == "__main__":
    train()

Writing /content/src/train.py


In [ ]:
!pip install -q albumentations tqdm

In [ ]:
import sys
sys.path.insert(0, "/content/src")
import torch
from model import ASLNet
from dataset import ASLDataset, build_transforms, CLASS_NAMES

# Model
m = ASLNet(num_classes=29, width=0.5)
out = m(torch.randn(2, 3, 128, 128))
print(f"✓ Model: {tuple(out.shape)} | {sum(p.numel() for p in m.parameters())/1e6:.2f}M params")

# Dataset
ds = ASLDataset("/content/dataset/asl_alphabet_train", build_transforms(False))
print(f"✓ Dataset: {len(ds)} images")

# Sample
img, lbl = ds[0]
print(f"✓ Sample: {tuple(img.shape)} → {CLASS_NAMES[lbl]}")

✓ Model: (2, 29) | 0.77M params
✓ Dataset: 87000 images
✓ Sample: (3, 128, 128) → A


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
from pathlib import Path
base = Path("/content/dataset")
target = base / "asl_alphabet_train"

def has_classes(p):
    if not p.exists(): return False
    names = set(d.name for d in p.iterdir() if d.is_dir())
    return {'A', 'B', 'space', 'del'}.issubset(names)

if target.exists() and (target / "asl_alphabet_train").exists() and has_classes(target / "asl_alphabet_train"):
    import shutil
    print("Flattening...")
    inner = target / "asl_alphabet_train"
    for item in inner.iterdir():
        shutil.move(str(item), str(target / item.name))
    inner.rmdir()
elif target.exists() and has_classes(target):
    print("✓ Already correct")
else:
    print("⚠️ Check structure:")
    for item in base.iterdir():
        print(" ", item.name)

✓ Already correct


In [ ]:
import shutil, os
os.makedirs("/content/drive/MyDrive/asl_project/src", exist_ok=True)
for f in ["model.py", "dataset.py", "train.py"]:
    src = f"/content/src/{f}"
    if os.path.exists(src):
        shutil.copy(src, f"/content/drive/MyDrive/asl_project/src/{f}")
        print(f"✓ backed up {f}")
    else:
        print(f"✗ missing {f}")

✓ backed up model.py
✓ backed up dataset.py
✓ backed up train.py


In [ ]:
import sys
sys.path.insert(0, "/content/src")

import importlib, dataset
importlib.reload(dataset)

from dataset import ASLDataset, build_transforms, CLASS_NAMES
print("Build transforms OK:", type(build_transforms(True)).__name__)

ds = ASLDataset("/content/dataset/asl_alphabet_train", build_transforms(True))
print(f"✓ Dataset: {len(ds)} images")
img, lbl = ds[0]
print(f"✓ Sample: {tuple(img.shape)} → {CLASS_NAMES[lbl]}")

Build transforms OK: Compose
✓ Dataset: 87000 images
✓ Sample: (3, 128, 128) → A


In [ ]:
!python /content/smoke_test.py

Device: cuda
Train batches: 1223 | Val batches: 136
Params: 0.77M
Epoch 1/2: 100% 1223/1223 [04:18<00:00,  4.72it/s, loss=1.5758, acc=60.38%]
  val: 100% 136/136 [00:11<00:00, 12.16it/s]
  → val_loss=3.3657  val_acc=3.61%
  ✓ saved (best=3.61%)
Epoch 2/2: 100% 1223/1223 [04:09<00:00,  4.90it/s, loss=0.5260, acc=95.59%]
  val: 100% 136/136 [00:11<00:00, 12.11it/s]
  → val_loss=3.3600  val_acc=4.18%
  ✓ saved (best=4.18%)

DONE. Best val acc: 4.18%


In [ ]:
import shutil, os
os.makedirs("/content/drive/MyDrive/asl_project/src", exist_ok=True)
for f in ["model.py", "dataset.py", "train.py"]:
    src = f"/content/src/{f}"
    if os.path.exists(src):
        shutil.copy(src, f"/content/drive/MyDrive/asl_project/src/{f}")
        print(f"✓ backed up {f}")
    else:
        print(f"✗ MISSING {f}")

# Also back up the smoke test script
if os.path.exists("/content/smoke_test.py"):
    shutil.copy("/content/smoke_test.py", "/content/drive/MyDrive/asl_project/src/smoke_test.py")
    print("✓ backed up smoke_test.py")

✓ backed up model.py
✓ backed up dataset.py
✓ backed up train.py
✓ backed up smoke_test.py


In [ ]:
%%writefile /content/full_train.py
import sys
sys.path.insert(0, "/content/src")
from train import train
train()

Writing /content/full_train.py


In [ ]:
!python /content/full_train.py

Device: cuda
Train batches: 1223 | Val batches: 136
Params: 0.77M
Epoch 1/30: 100% 1223/1223 [04:11<00:00,  4.87it/s, loss=1.6443, acc=57.91%]
  val: 100% 136/136 [00:10<00:00, 12.43it/s]
  → val_loss=3.3664  val_acc=3.67%
  ✓ saved (best=3.67%)
Epoch 2/30: 100% 1223/1223 [04:10<00:00,  4.87it/s, loss=0.5679, acc=93.98%]
  val: 100% 136/136 [00:11<00:00, 12.26it/s]
  → val_loss=3.3635  val_acc=5.09%
  ✓ saved (best=5.09%)
Epoch 3/30: 100% 1223/1223 [04:16<00:00,  4.77it/s, loss=0.4910, acc=96.32%]
  val: 100% 136/136 [00:11<00:00, 12.27it/s]
  → val_loss=3.3668  val_acc=4.14%
Epoch 4/30: 100% 1223/1223 [04:14<00:00,  4.81it/s, loss=0.4598, acc=97.18%]
  val: 100% 136/136 [00:10<00:00, 12.41it/s]
  → val_loss=3.3760  val_acc=3.39%
Epoch 5/30: 100% 1223/1223 [04:12<00:00,  4.83it/s, loss=0.4404, acc=97.76%]
  val: 100% 136/136 [00:11<00:00, 12.00it/s]
  → val_loss=3.3990  val_acc=3.38%
Epoch 6/30: 100% 1223/1223 [04:11<00:00,  4.87it/s, loss=0.4278, acc=98.11%]
  val: 100% 136/136 [00:11

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA: True
GPU: Tesla T4


In [ ]:
import os, torch

ckpt_path = "/content/drive/MyDrive/asl_project/checkpoints/best.pt"
print("Exists:", os.path.exists(ckpt_path))
if os.path.exists(ckpt_path):
    print("Size MB:", round(os.path.getsize(ckpt_path)/1e6, 2))
    ckpt = torch.load(ckpt_path, map_location="cpu")
    print("Keys:", list(ckpt.keys()))
    print("Saved val_acc (broken EMA):", ckpt.get("val_acc"))
else:
    print("❌ No checkpoint on Drive — must retrain")

Exists: True
Size MB: 6.39
Keys: ['ema_state', 'model_state', 'config', 'val_acc']
Saved val_acc (broken EMA): 5.091954022988506


In [ ]:
!mkdir -p /content/dataset
!unzip -q /content/drive/MyDrive/asl_project/archive.zip -d /content/dataset
print("✓ Extracted")

replace /content/dataset/asl_alphabet_test/asl_alphabet_test/A_test.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: ✓ Extracted


In [ ]:
import shutil, os
os.makedirs("/content/src", exist_ok=True)
for f in ["model.py", "dataset.py", "train.py"]:
    src = f"/content/drive/MyDrive/asl_project/src/{f}"
    if os.path.exists(src):
        shutil.copy(src, f"/content/src/{f}")
        print(f"✓ restored {f}")
    else:
        print(f"✗ MISSING: {f}")

✓ restored model.py
✓ restored dataset.py
✓ restored train.py


In [ ]:
import sys
sys.path.insert(0, "/content/src")
import torch
from tqdm import tqdm
from model import ASLNet
from dataset import make_loaders, CLASS_NAMES

model = ASLNet(num_classes=29, width=0.5)

ckpt = torch.load(
    "/content/drive/MyDrive/asl_project/checkpoints/best.pt",
    map_location="cpu"
)
model.load_state_dict(ckpt["model_state"])   # ← the REAL model, not EMA
model = model.cuda().eval()
print("✓ Loaded model_state")

_, val_loader = make_loaders("/content/dataset/asl_alphabet_train", 128, 64, 2)
correct, total = 0, 0
with torch.no_grad():
    for x, y in tqdm(val_loader, desc="Evaluating"):
        x, y = x.cuda(), y.cuda()
        logits = model(x)
        correct += (logits.argmax(1) == y).sum().item()
        total += x.size(0)

acc = 100 * correct / total
print(f"\n🎯 RECOVERED model val_acc: {acc:.2f}%")

✓ Loaded model_state


Evaluating: 100%|██████████| 136/136 [00:11<00:00, 11.69it/s]


🎯 RECOVERED model val_acc: 98.87%


In [ ]:
import torch, os

save_path = "/content/drive/MyDrive/asl_project/checkpoints/model_real.pt"
torch.save({
    "model_state": model.state_dict(),   # 98.87% weights
    "val_acc": acc,
    "config": {"img_size": 128, "num_classes": 29, "width": 0.5},
    "class_names": CLASS_NAMES,
}, save_path)

print(f"✓ Saved to {save_path}")
print(f"✓ Size: {os.path.getsize(save_path)/1e6:.2f} MB")

✓ Saved to /content/drive/MyDrive/asl_project/checkpoints/model_real.pt
✓ Size: 3.21 MB


In [ ]:
from google.colab import files
files.download("/content/drive/MyDrive/asl_project/checkpoints/model_real.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
p = "/content/drive/MyDrive/asl_project/checkpoints/model_real.pt"
print("On Drive:", os.path.exists(p))
print("Size MB:", round(os.path.getsize(p)/1e6, 2))

On Drive: True
Size MB: 3.21
